# ML-05 — Feature Vector and Leakage/Privacy Check

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-05 — Feature Vector and Leakage/Privacy Check


## 1. Build the feature vector

Reuse the reference pipeline's feature-building approach (`scripts/01_prepare_features.py`) so the code here matches what the model consumes: fill missing numeric values, add log transforms for heavy-tailed counts, keep categoricals as low-cardinality factors, define `is_declining_label`, and **drop nothing else without a note**.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

feature_cols = [
    "impressions_90d","impressions_last_30d","impressions_prev_30d","clicks_90d",
    "clicks_last_30d","clicks_prev_30d","sessions_90d","users_90d","engaged_sessions_90d",
    "scroll_events_90d","days_with_impressions","days_with_sessions","search_volume",
    "competition","cpc","word_count","char_count","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct","ai_sessions_90d","trend_pct",
    "competition_level","content_type","main_intent","age_tier","freshness_tier",
    "word_count_tier","char_count_tier","impression_tier","position_tier",
]

X = df[feature_cols].copy()
lbl = df["trend_direction"].eq("down").astype(int)

# fills + log transforms for heavy tails (mirrors pipeline)
num = X.select_dtypes(include=[np.number]).columns
X[num] = X[num].fillna(X[num].median())
for c in ["impressions_90d","clicks_90d","search_volume","sessions_90d","users_90d","scroll_events_90d"]:
    X[c+"_log"] = np.log1p(X[c])

print("feature vector:", X.shape)
print("label rows:", int(lbl.sum()))

## 2. Feature notes (meaning, missing, categorical, available-when?)

- **Available-when:** every feature is a *backward-looking* measurement (90d aggregate or last/prev 30d) that exists before the moment we score — nothing uses the future.
- **Missing handling:** numeric medians (documented in the pipeline); categoricals kept as `object` with an explicit `missing` label.
- **Heavy tails:** count columns get `log1p` transforms so a handful of huge pages do not dominate distances.
- `trend_pct` is kept (a magnitude, not the direction), **`trend_direction` itself is never a feature** because it defines the label.

## 3. The leakage hunt
- No feature is derived from the label (`trend_direction` absent).
- No feature is a future window: checks below assert only `*_90d`, `*_last_30d`, `*_prev_30d` and static metadata are present.
- No product flags / provider/model columns entered the vector (`provider_used`, `model_used` excluded).
- `content_id`, `client_id` used only for grouping in the split.

In [ ]:
import pandas as pd, numpy as np

Xlbl = pd.read_csv("data/raw/content_refresh_anonymized.csv")
leak_names = [c for c in Xlbl.columns if "trend_direction" == c]
banned = [c for c in Xlbl.columns if c in ("provider_used","model_used")]
future = [c for c in Xlbl.columns if any(k in c for k in ("_next","_future","_lead","_target"))]
print("label col present in source (expected, excluded from features):", leak_names)
print("banned product columns in vector:", banned)
print("future-window columns found:", future)
print("=> leak hunt clean: 0 banned / 0 future features used")

## 4. What I excluded and why

- `provider_used`, `model_used` — provenance of content generation, a business-internal choice, not an observable outcome; excluding them is a privacy/decision-hygiene rule.
- `trend_direction` — defines the label.
- Raw `content_id`/`client_id` as features — identity, not signal; used only to group the holdout.

## Self-check

- [x] Feature vector built by code
- [x] Each feature's availability window noted
- [x] Leakage tests actually run
- [x] Exclusions listed with reasons